# First prototype for HyperBubble Resolution-Oriented GNN

In [13]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import json, math, time, random, re, glob, os, itertools
import numpy as np
import pandas as pd

USE_DIRECTML = True

import torch
device = torch.device('cpu')
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device('cpu')

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

import matplotlib.pyplot as plt

Using DirectML: privateuseone:0


In [14]:
NUC2ID = {'A':1, 'C':2, 'G':3, 'T':4, 'N':5}

def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def tokens_to_onehot(tokens: torch.LongTensor, num_classes: int = 6):
    N, K = tokens.shape
    classes = torch.arange(num_classes, device=tokens.device).view(1, 1, num_classes)
    return (tokens.unsqueeze(-1) == classes).float()

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())


In [15]:
import json
from pathlib import Path
from typing import List, Dict, Any, Optional
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data

class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset with token sequences for CNN encoding.
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float]=None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        for n in _safe_get(rec,"nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"upstream_nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"downstream_nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"inside_nodes",[]):
            if isinstance(n,dict) and "seq" in n:
                ensure_node(n["seq"], n.get("cov"))

        ensure_node(start_seq)
        ensure_node(end_seq)

        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec,"edges",[]):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes",0)),
                float(e.get("len_bp",0)),
                float(e.get("cov_min",0)),
                float(e.get("cov_mean",0.0)),
                1.0 if e.get("on_ref",False) else 0.0
            ])

        node_order = [None]*len(node_seqs)
        for s,i in node_seqs.items():
            node_order[i]=s

        # store tokens only, CNN handles one-hot
        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)

        start_idx = torch.tensor(node_seqs[start_seq])
        end_idx   = torch.tensor(node_seqs[end_seq])

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float32) if edge_attr else torch.zeros((0,5))

        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1

        paths_list = _safe_get(rec,"paths",[])
        lp = rec.get("label_path")
        if lp is not None and 0 <= lp < len(paths_list):
            label_path_idx = lp
            if num_edges > 0:
                y_edge_mask[torch.tensor(paths_list[lp], dtype=torch.long)] = 1

        data = Data(
            seq_tokens=seq_tokens,
            x_cov=x_cov,
            edge_index=edge_index,
            edge_attr=edge_attr,
            start_idx=start_idx,
            end_idx=end_idx,
            num_nodes=seq_tokens.size(0),
            y_edge_mask=y_edge_mask,
            label_path_idx=torch.tensor(label_path_idx),
        )
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"])
        if "k" in rec:
            data.k = torch.tensor(rec["k"])
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])

In [16]:
# [DISCOVERY] automatyczne wyszukiwanie plików z bąblami po pokryciu i genomie

# katalogi, w których leżą dane
SEARCH_DIRS = [
    Path("lower_cov_data"),
    Path("other_cov_data")
]

def discover_base_datasets_for_cov(coverage: int) -> Dict[str, str]:
    """
    Znajdź wszystkie pliki w formacie:
        {genome_id}_dataset_cov{coverage}_ratio_lt_3.5.jsonl
    w katalogach z SEARCH_DIRS.

    Zwraca słownik:
        {genome_id: pełna_ścieżka_do_pliku}
    """
    pattern = f"*dataset_cov{coverage}_ratio_lt_3.5.jsonl"
    candidates: List[Path] = []

    for d in SEARCH_DIRS:
        if not d.is_dir():
            continue
        candidates.extend(d.glob(pattern))

    # unikalne + posortowane
    unique = sorted({p.resolve() for p in candidates})

    if not unique:
        raise FileNotFoundError(
            f"Nie znaleziono plików dla pokrycia {coverage} (wzorzec: {pattern}) "
            f"w katalogach: {', '.join(str(d) for d in SEARCH_DIRS)}"
        )

    mapping: Dict[str, str] = {}
    for p in unique:
        name = p.name
        # genome_id = wszystko przed '_dataset_cov'
        genome_id = name.split("_dataset_cov")[0]
        mapping[genome_id] = str(p)

    print(f"[discover_base_datasets_for_cov] coverage={coverage} -> znaleziono {len(mapping)} genomów:")
    return mapping


COVERAGES = [20]
COV_TO_FILES = {cov: discover_base_datasets_for_cov(cov) for cov in COVERAGES}

# zbiór wszystkich genomów, jakie realnie istnieją w danych
ALL_GENOMES = sorted(set().union(*[m.keys() for m in COV_TO_FILES.values()]))
print("ALL_GENOMES:", ALL_GENOMES)


# Keep only labeled samples
def labeled_subset(ds):
    labeled_indices = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            labeled_indices.append(i)
    return Subset(ds, labeled_indices)


[discover_base_datasets_for_cov] coverage=20 -> znaleziono 10 genomów:
ALL_GENOMES: ['Bacillus', 'Citrobacter', 'Cronobacter', 'Ecoli', 'Haemophilus', 'Klebsiella', 'Morganella', 'Proteus', 'Salmonella', 'Serratia']


In [26]:
class HyperbubbleGNN(nn.Module):
    """
    CNN-based sequence encoder + 2× GCNConv + MLP edge classifier.
    Designed to be DirectML-friendly (no dense scatter ops).
    """
    def __init__(
        self,
        vocab_size=6,        # one-hot channels
        cnn_channels=32,     # Conv1d output channels
        gcn_hidden=64,
        edge_hidden=64,
        edge_feat_dim=5,
        dropout=0.0,
    ):
        super().__init__()

        # 1-hot CNN encoder over k-mer tokens
        self.cnn = nn.Conv1d(vocab_size, cnn_channels, kernel_size=3, padding=1)

        # node feature = CNN embedding + coverage
        self.node_in = cnn_channels + 1

        # sparse GCN layers (operate directly on edge_index)
        self.gcn1 = GCNConv(self.node_in, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        self.dropout = nn.Dropout(dropout)
        self.gcn_hidden = gcn_hidden

        # edge classifier MLP
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] (Long, k-mer tokens)
        x_cov:      [N, 1] (Float, node coverage)
        returns:    [N, cnn_channels + 1]
        """
        # NOTE: tokens_to_onehot should be implemented in a DML-safe way
        # e.g. using broadcasting instead of F.one_hot.
        onehot = tokens_to_onehot(seq_tokens)      # [N, K, vocab_size]
        x = onehot.permute(0, 2, 1)                # [N, vocab_size, K]

        H = F.relu(self.cnn(x))                    # [N, C, K]
        H = H.mean(dim=2)                          # [N, C] global pooling

        return torch.cat([H, x_cov], dim=1)        # [N, C+1]

    def forward(self, data):
        """
        data:
          - seq_tokens [N,K]
          - x_cov      [N,1]
          - edge_index [2,E]
          - edge_attr  [E,5] (optional)
          - batch      [N]   (optional, for multi-graph batches)
        returns:
          - logits [E]
        """
        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)  # [N, node_in]

        # GCN over whole batch (PyG handles 'batch' internally if present)
        H = F.relu(self.gcn1(X0, data.edge_index))
        H = self.dropout(H)

        H = F.relu(self.gcn2(H, data.edge_index))
        H = self.dropout(H)

        # edge-wise features
        u, v = data.edge_index
        U = H[u]
        V = H[v]

        if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0:
            EA = data.edge_attr
        else:
            E = data.edge_index.size(1)
            EA = H.new_zeros((E, 5))

        feats = torch.cat([U, V, EA], dim=1)       # [E, 2*gcn_hidden + 5]
        logits = self.edge_mlp(feats).squeeze(-1)  # [E]

        return logits

In [16]:
def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    """
    Return list of (cand_idx_tensor, gold_pos_tensor) for EVERY decision node
    whose source node appears on the labeled path. This is orientation-agnostic.
    """
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

def train_one_epoch_choice_all(model, loader, optim, device):
    model.train()
    total_loss, total_decisions = 0.0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        batch_loss = 0.0
        dec_used = 0
        batch_correct = 0        # NEW
        batch_total = 0          # NEW

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            for cand_idx, gold_pos in decisions:

                # compute loss
                ce = F.cross_entropy(logits[cand_idx].unsqueeze(0), gold_pos.view(1))
                batch_loss += ce
                dec_used += 1

                # compute train-time accuracy
                pred = logits[cand_idx].argmax().item()
                gold = gold_pos.item()
                batch_correct += int(pred == gold)
                batch_total += 1

        if dec_used == 0:
            continue

        batch_loss = batch_loss / dec_used
        optim.zero_grad()
        batch_loss.backward()
        optim.step()

        total_loss += batch_loss.item()
        total_decisions += dec_used

        # ---- PRINT TRAIN BATCH ACCURACY ----
        if batch_total > 0:
            acc = batch_correct / batch_total
            print(f"[train] batch_decisions={batch_total} | batch_acc={acc:.3f}")

    return total_loss / max(total_decisions, 1)


def _cpu_state_dict(sd):
    return {k: v.detach().cpu().clone() for k, v in sd.items()}

best_ckpt = {
    "epoch": 0,
    "dec_acc": -1.0,
    "bub_acc": -1.0,
    "state_dict": None,
}

def _is_better(bub_acc, dec_acc, best):
    if bub_acc > best["bub_acc"]:
        return True
    if bub_acc == best["bub_acc"] and dec_acc > best["dec_acc"]:
        return True
    return False

@torch.no_grad()
def eval_choice_all(model, loader, device):
    model.eval()
    total_decisions, correct_decisions = 0, 0
    total_bubbles, correct_bubbles = 0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            if not decisions:
                continue
            total_bubbles += 1

            all_correct = True
            for cand_idx, gold_pos in decisions:
                pred = logits[cand_idx].argmax().item()
                ok = int(pred == gold_pos.item())
                correct_decisions += ok
                total_decisions += 1
                all_correct &= bool(ok)

            correct_bubbles += int(all_correct)

    dec_acc = correct_decisions / max(total_decisions, 1)
    bub_acc = correct_bubbles / max(total_bubbles, 1)
    print(f"[choice-eval] decisions={total_decisions} acc_dec={dec_acc:.3f} | bubbles={total_bubbles} acc_bub={bub_acc:.3f}")
    return dec_acc, bub_acc

In [17]:
OUTDIR = Path("sweep_from_paths_more_genomes"); OUTDIR.mkdir(exist_ok=True, parents=True)

SEED = 42
BATCH_SIZE = 64
NUM_WORKERS = 0
EPOCHS = 200
PATIENCE = 100
LR = 1e-3
WEIGHT_DECAY = 0.0
GRAD_CLIP = 1.0
VAL_FRAC = 0.1  # fraction of TRAIN reserved for validation

EMBED_GRID = [16, 32]
GCN_HIDDEN_GRID = [64, 128]
EDGE_HIDDEN_GRID = [64]
USE_LSTM_OPTS = [False]

PRINT_EVERY = 1

In [18]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def split_train_val(n: int, val_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    order = np.arange(n)
    rng.shuffle(order)
    n_val = int(round(val_frac * n))
    val_idx = order[:n_val]
    tr_idx  = order[n_val:]
    return tr_idx.tolist(), val_idx.tolist()

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    """
    Zwraca listę (cand_idx, gold_pos) dla każdego węzła źródłowego w grafie g,
    dla którego istnieje decyzja (zaznaczone krawędzie y_edge_mask==1).
    """
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

@torch.no_grad()
def _candidate_sets(data, logits: torch.Tensor):
    """
    Zwraca listę (cand_idx [M], gold_pos [1], g) dla każdej decyzji w batchu.
    """
    u = data.edge_index[0]
    node_batch = (
        data.batch if hasattr(data, "batch")
        else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    )
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)
    out = []
    for g in range(num_graphs):
        decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in decisions:
            out.append((cand_idx, gold_pos, g))
    return out

def decision_ce_loss_from_batch(logits: torch.Tensor, data, reduction: str = "mean"):
    """
    Cross-entropy liczona po decyzjach (candidate sets).
    """
    if logits.numel() == 0:
        return logits.new_zeros(()), 0
    losses, decisions = [], 0
    u = data.edge_index[0]
    node_batch = (
        data.batch if hasattr(data, "batch")
        else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    )
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)

    for g in range(num_graphs):
        cands = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in cands:
            cand_logits = logits[cand_idx]     # [M]
            target = gold_pos.view(1)          # [1]
            losses.append(F.cross_entropy(cand_logits.unsqueeze(0), target))
            decisions += 1

    if decisions == 0:
        return logits.new_zeros(()), 0
    loss = torch.stack(losses)
    return (loss.mean() if reduction == "mean" else loss.sum()), decisions


@torch.no_grad()
def eval_rich_metrics(model, loader, device) -> Dict[str, float]:
    """
    Minimalny zestaw metryk:
      - acc_dec   : dokładność decyzji (wybór ścieżki)
      - acc_bub   : dokładność bąbli (wszystkie decyzje w bąblu poprawne)
      - precision_dec, recall_dec, f1_dec : metryki binarne dla decyzji
      - brier     : błąd Briera dla prawdopodobieństwa poprawnej ścieżki

    Wszystko w oparciu o candidate sets, bez ECE / ROC / PR-AUC / MRR.
    """
    model.eval()
    dec_correct, dec_total = 0, 0
    bubble_ok = {}  # key: (id(data), g)

    confs_gold = []   # p(model poprawny) dla poprawnej ścieżki (gold)
    labels_gold = []  # 1/0 czy decyzja faktycznie poprawna

    y_true_dec = []   # 1 jeśli decyzja poprawna
    y_pred_dec = []   # 1 jeśli model jest "pewny" (p(pred) >= 0.5)

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        for cand_idx, gold_pos, g in _candidate_sets(data, logits):
            cand_logits = logits[cand_idx]              # [M]
            probs = F.softmax(cand_logits, dim=0)
            pred = int(torch.argmax(cand_logits).item())
            gold = int(gold_pos.item())

            ok = int(pred == gold)
            dec_total += 1
            dec_correct += ok

            key = (id(data), int(g))
            bubble_ok.setdefault(key, True)
            bubble_ok[key] = bubble_ok[key] and bool(ok)

            # Brier: prawdopodobieństwo poprawnej ścieżki
            conf_gold = float(probs[gold].item())
            confs_gold.append(conf_gold)
            labels_gold.append(float(ok))

            # klasyfikacja binarna dla precision/recall/F1:
            #   true = decyzja poprawna
            #   pred = pewny (p(pred) >= 0.5)
            y_true_dec.append(float(ok))
            conf_pred = float(probs[pred].item())
            y_pred_dec.append(1.0 if conf_pred >= 0.5 else 0.0)

    acc_dec = dec_correct / max(dec_total, 1)

    bub_total = len(bubble_ok)
    bub_correct = sum(1 for ok in bubble_ok.values() if ok)
    acc_bub = bub_correct / max(bub_total, 1)

    # Brier
    confs_gold = np.array(confs_gold, dtype=np.float64)
    labels_gold = np.array(labels_gold, dtype=np.float64)
    if confs_gold.size:
        brier = float(((confs_gold - labels_gold) ** 2).mean())
    else:
        brier = float("nan")

    # precision / recall / F1 dla decyzji
    y_true_dec = np.array(y_true_dec, dtype=np.int32)
    y_pred_dec = np.array(y_pred_dec, dtype=np.int32)
    if y_true_dec.size:
        tp = int(((y_true_dec == 1) & (y_pred_dec == 1)).sum())
        fp = int(((y_true_dec == 0) & (y_pred_dec == 1)).sum())
        fn = int(((y_true_dec == 1) & (y_pred_dec == 0)).sum())

        precision_dec = tp / max(tp + fp, 1)
        recall_dec    = tp / max(tp + fn, 1)
        if precision_dec + recall_dec > 0:
            f1_dec = 2 * precision_dec * recall_dec / (precision_dec + recall_dec)
        else:
            f1_dec = 0.0
    else:
        precision_dec = recall_dec = f1_dec = float("nan")

    return {
        "acc_dec": acc_dec,
        "acc_bub": acc_bub,
        "precision_dec": precision_dec,
        "recall_dec": recall_dec,
        "f1_dec": f1_dec,
        "brier": brier,
        "decisions": dec_total,
        "bubbles": bub_total,
    }


def train_one_epoch(model, loader, device, optimizer, grad_clip: Optional[float] = 1.0):
    model.train()
    total_loss, total_decisions = 0.0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(data)
        loss, ndec = decision_ce_loss_from_batch(logits, data, reduction="mean")
        if ndec == 0:
            continue
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += float(loss.item()) * ndec
        total_decisions += ndec
    return total_loss / max(total_decisions, 1)


def fit_variant(build_model_fn,
                train_loader,
                val_loader,
                device,
                out_ckpt: Path,
                epochs=20,
                lr=1e-3,
                weight_decay=0,
                patience=5,
                grad_clip=1.0,
                seed=42,
                print_every=5,
                early_stop=True):
    set_seed(seed)
    model = build_model_fn().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = -1.0
    best_state = None
    waited = 0
    history = []

    for ep in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, device, opt, grad_clip)
        val_stats = eval_rich_metrics(model, val_loader, device)
        history.append({"epoch": ep, "train_loss": tr_loss, **val_stats})
        metric = val_stats["acc_bub"]

        if (ep % print_every == 0) or (ep == 1):
            print(
                f"[ep {ep:03d}] train_loss={tr_loss:.4f}  "
                f"val_bub={val_stats['acc_bub']:.4f}  "
                f"val_dec={val_stats['acc_dec']:.4f}"
            )

        if metric > best_val:
            best_val = metric
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            waited = 0
        else:
            waited += 1

        if early_stop and waited >= patience:
            break

    if best_state is None:
        best_state = model.state_dict()
    torch.save(best_state, out_ckpt)
    return best_state, pd.DataFrame(history)


In [19]:
COVERAGE_SWEEP = 20
GENOME_TO_PATH = COV_TO_FILES[COVERAGE_SWEEP]  # {genome_id: ścieżka}
GENOMES = sorted(GENOME_TO_PATH.keys())
print("Genomy używane w sweepie architektur:", GENOMES)

OUTDIR = Path("sweep_from_paths_more_genomes")
OUTDIR.mkdir(parents=True, exist_ok=True)

Genomy używane w sweepie architektur: ['Bacillus', 'Citrobacter', 'Cronobacter', 'Ecoli', 'Haemophilus', 'Klebsiella', 'Morganella', 'Proteus', 'Salmonella', 'Serratia']


In [27]:
# %% sweep runner — k-fold (leave-one-genome-out) across model sizes

rows = []
set_seed(SEED)

for ed in EMBED_GRID:
    for gh in GCN_HIDDEN_GRID:
        for eh in EDGE_HIDDEN_GRID:
            for ul in USE_LSTM_OPTS:
                variant = f"emb{ed}_g{gh}_e{eh}_{'lstm' if ul else 'mean'}"
                print(f"\n=== SWEEP: {variant} ===")

                def build_model():
                    return HyperbubbleGNN(
                        vocab_size=6,
                        cnn_channels=16,
                        gcn_hidden=gh,
                        edge_hidden=eh,
                        edge_feat_dim=5,
                        dropout=0.0
                    )

                for fold, test_genome in enumerate(GENOMES, start=1):
                    test_paths  = [GENOME_TO_PATH[test_genome]]
                    train_paths = [GENOME_TO_PATH[g] for g in GENOMES if g != test_genome]

                    # datasety
                    train_full = HyperbubbleDataset(train_paths)
                    test_full  = HyperbubbleDataset(test_paths)

                    train_lab = labeled_subset(train_full)
                    test_lab  = labeled_subset(test_full)
                    if len(train_lab) == 0 or len(test_lab) == 0:
                        print(f"[WARN] brak etykiet dla test_genome={test_genome}, pomijam fold")
                        continue

                    tr_idx, va_idx = split_train_val(len(train_lab), VAL_FRAC, SEED)
                    train_ds = Subset(train_lab, tr_idx)
                    val_ds   = Subset(train_lab, va_idx)
                    test_ds  = test_lab

                    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
                    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
                    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

                    print(
                        f"[{variant}] fold={fold} test={test_genome} "
                        f"(train={len(train_ds)} val={len(val_ds)} test={len(test_ds)})"
                    )

                    ckpt    = OUTDIR / f"{variant}_cov{COVERAGE_SWEEP}_fold{fold}_{test_genome}.pth"
                    log_csv = OUTDIR / f"{variant}_cov{COVERAGE_SWEEP}_fold{fold}_{test_genome}_history.csv"

                    best_state, hist_df = fit_variant(
                        build_model_fn=build_model,
                        train_loader=train_loader,
                        val_loader=val_loader,
                        device=device,
                        out_ckpt=ckpt,
                        epochs=EPOCHS,
                        lr=LR,
                        weight_decay=WEIGHT_DECAY,
                        patience=PATIENCE,
                        grad_clip=GRAD_CLIP,
                        seed=SEED,
                        print_every=PRINT_EVERY,
                        early_stop=True,
                    )
                    hist_df["variant"]     = variant
                    hist_df["coverage"]    = COVERAGE_SWEEP
                    hist_df["fold"]        = fold
                    hist_df["test_genome"] = test_genome
                    hist_df.to_csv(log_csv, index=False)

                    model = build_model().to(device)
                    model.load_state_dict(best_state)
                    model.eval()
                    stats = eval_rich_metrics(model, test_loader, device)
                    stats.update({
                        "variant": variant,
                        "coverage": COVERAGE_SWEEP,
                        "fold": fold,
                        "test_genome": test_genome,
                        "params": count_params(model),
                    })
                    rows.append(stats)

                    print(
                        f"[TEST] {variant} | fold={fold} | test={test_genome} -> "
                        f"acc_bub={stats['acc_bub']:.3f}  "
                        f"acc_dec={stats['acc_dec']:.3f}  "
                        f"F1_dec={stats['f1_dec']:.3f}"
                    )

df = (
    pd.DataFrame(rows)
      .sort_values(by=["variant", "test_genome", "acc_bub"], ascending=[True, True, False])
)
df_path = OUTDIR / "sweep_results.csv"
df.to_csv(df_path, index=False)
print(f"\nSaved: {df_path}")
df.head(12)


=== SWEEP: emb16_g64_e64_mean ===
[emb16_g64_e64_mean] fold=1 test=Bacillus (train=2435 val=271 test=132)


RuntimeError: Parametr jest niepoprawny.

In [ ]:

results_path = OUTDIR / "sweep_results.csv"
df = pd.read_csv(results_path)

# --- 1. Agregacja po wariancie (średnio po genomach/foldach) ---

agg = (
    df.groupby("variant")
      .agg(
          acc_bub_mean=("acc_bub", "mean"),
          acc_dec_mean=("acc_dec", "mean"),
          f1_dec_mean=("f1_dec", "mean"),
          precision_dec_mean=("precision_dec", "mean"),
          recall_dec_mean=("recall_dec", "mean"),
          brier_mean=("brier", "mean"),
          params=("params", "first"),
      )
      .reset_index()
      .sort_values("acc_bub_mean", ascending=False)
)

agg_path = OUTDIR / "summary_architectures_kfold.csv"
agg.to_csv(agg_path, index=False)
print("Zapisano podsumowanie architektur:", agg_path)

# --- 2. Podział na małe / duże modele (wg mediany #param) ---

median_params = agg["params"].median()
small = agg[agg["params"] <= median_params].copy()
large = agg[agg["params"] > median_params].copy()

def plot_arch_bar(df_part: pd.DataFrame, title: str, fname: str):
    if df_part.empty:
        print(f"[WARN] brak modeli do narysowania dla: {title}")
        return
    ax = df_part.set_index("variant")[["acc_bub_mean", "acc_dec_mean"]].plot(
        kind="bar", rot=45, figsize=(10, 4)
    )
    ax.set_ylabel("Dokładność")
    ax.set_title(title)
    ax.legend(["Dokładność bąbli", "Dokładność decyzji"])
    ax.figure.tight_layout()
    ax.figure.savefig(OUTDIR / fname, dpi=200)
    plt.close(ax.figure)

plot_arch_bar(
    small,
    "Porównanie architektur (małe modele, k-fold po genomach)",
    "arch_kfold_acc_small.png",
)
plot_arch_bar(
    large,
    "Porównanie architektur (duże modele, k-fold po genomach)",
    "arch_kfold_acc_large.png",
)

# --- 3. Zależność dokładności od liczby parametrów ---

df_sc = agg.sort_values("params")
plt.figure(figsize=(7, 4))
plt.plot(df_sc["params"], df_sc["acc_bub_mean"], marker="o", label="Dokładność bąbli")
plt.plot(df_sc["params"], df_sc["acc_dec_mean"], marker="s", label="Dokładność decyzji")
plt.xlabel("Liczba parametrów modelu")
plt.ylabel("Dokładność (średnia)")
plt.title("Dokładność vs liczba parametrów (k-fold po genomach)")
plt.legend(loc="best")
plt.tight_layout()
plt.savefig(OUTDIR / "arch_kfold_params_vs_acc.png", dpi=200)
plt.close()

# --- 4. Najlepszy wariant: metryki na poszczególnych genomach ---

best_variant = agg.iloc[0]["variant"]
print("Najlepszy wariant (wg acc_bub_mean):", best_variant)

best_per_genome = (
    df[df["variant"] == best_variant]
      .groupby("test_genome")
      .agg(
          acc_bub=("acc_bub", "mean"),
          acc_dec=("acc_dec", "mean"),
          f1_dec=("f1_dec", "mean"),
          precision_dec=("precision_dec", "mean"),
          recall_dec=("recall_dec", "mean"),
          brier=("brier", "mean"),
      )
      .reset_index()
      .sort_values("acc_bub", ascending=False)
)

best_genome_csv = OUTDIR / f"{best_variant}_per_genome_kfold.csv"
best_per_genome.to_csv(best_genome_csv, index=False)
print("Zapisano per-genome dla najlepszego wariantu:", best_genome_csv)

# wykres: acc_bub / acc_dec / F1 po genomach
ax = best_per_genome.set_index("test_genome")[["acc_bub", "acc_dec", "f1_dec"]].plot(
    kind="bar", rot=0, figsize=(8, 4)
)
ax.set_ylabel("Dokładność")
ax.set_title(f"Wyniki najlepszego wariantu na poszczególnych genomach\n({best_variant}, k-fold)")
ax.legend(["Dokładność bąbli", "Dokładność decyzji", "F1 (decyzje)"])
ax.figure.tight_layout()
ax.figure.savefig(OUTDIR / f"{best_variant}_per_genome_kfold.png", dpi=200)
plt.close(ax.figure)

print("Wykresy zapisane w katalogu:", OUTDIR.resolve())

In [ ]:
# # Where to look for the datasets (add paths if needed)
# SEARCH_DIRS = [
#     ".", 
#     "other_cov_data",
#     "lower_cov_data",
# ]

# COVERAGES = [10, 20, 30]   # target coverage magnitudes
# GENOME_KEYS = ["ecoli", "klebsiella", "salmonella"]  # expected 3 genomes

# # Sweep (non-LSTM only here)
# EMBED_GRID = [16, 32]
# GCN_HIDDEN_GRID = [32, 64, 128]
# EDGE_HIDDEN_GRID = [16, 32, 64]
# USE_LSTM_OPTS = [False]    # IMPORTANT: no LSTM locally

# # Train/validation/test split within each coverage
# VAL_FRAC = 0.1
# SEED = 42
# PRINT_EVERY = 5

# # I/O
# OUTDIR = Path("cov_sweep_runs"); OUTDIR.mkdir(parents=True, exist_ok=True)
# AGG_PATH = OUTDIR / "cov_all_results.csv"

# print("Device:", device)
# print("Output dir:", OUTDIR.resolve())


In [ ]:
# def _match_cov_files(coverage: int):
#     """
#     Return dict {genome -> path} for a given coverage.
#     Accept patterns:
#       *_cov{coverage}*_ratio_lt_3.5.jsonl
#       *_cov{coverage}*_ratio_lt_35.jsonl
#       *_cov{coverage}*.jsonl
#     Prefers files with ratio suffix if multiple exist.
#     """
#     patterns = [
#         f"*cov{coverage}*ratio*3.5*.jsonl",  # ratio_lt_3.5.*
#         f"*cov{coverage}*ratio*35*.jsonl",   # ratio_lt_35.*
#         f"*cov{coverage}*.jsonl",            # fallback
#     ]
#     candidates = []
#     for d in SEARCH_DIRS:
#         for pat in patterns:
#             candidates.extend([str(p) for p in Path(d).glob(pat)])
#     # unique by path
#     candidates = list(dict.fromkeys(candidates))

#     def genome_key_from_path(p: str):
#         name = Path(p).name.lower()
#         for g in GENOME_KEYS:
#             if g in name:
#                 return g
#         # fallback: prefix before first underscore
#         return name.split("_")[0]

#     # keep the most specific matches first (ratio name preferred)
#     def score(p: str):
#         name = Path(p).name
#         s = 0
#         if "ratio" in name: s += 2
#         if re.search(r"3[._]?5", name): s += 1
#         return -s  # sort asc by negative score => higher score first

#     candidates.sort(key=score)

#     mapping = {}
#     for p in candidates:
#         g = genome_key_from_path(p)
#         if g in GENOME_KEYS and g not in mapping:
#             mapping[g] = p
#         if len(mapping) == len(GENOME_KEYS):
#             break
#     return mapping

# COV_TO_FILES = {cov: _match_cov_files(cov) for cov in COVERAGES}

# # sanity checks & print
# for cov, mp in COV_TO_FILES.items():
#     print(f"Coverage {cov}:")
#     for g in GENOME_KEYS:
#         print(f"  {g:12s} -> {mp.get(g, 'MISSING')}")
#     if any(g not in mp for g in GENOME_KEYS):
#         raise FileNotFoundError(f"Missing genome files for cov{cov}. Found: {mp}")


In [ ]:
# # Fixed best-variant configuration
# BEST_EMB = 16
# BEST_GCN = 64
# BEST_EDGE = 64
# USE_LSTM = False
# VARIANT = f"emb{BEST_EMB}_g{BEST_GCN}_e{BEST_EDGE}_{'lstm' if USE_LSTM else 'mean'}"

# def set_seed(seed: int):
#     import random
#     random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
#     if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# def labeled_subset(ds):
#     idx = []
#     for i in range(len(ds)):
#         d = ds[i]
#         lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
#         if lp >= 0:
#             idx.append(i)
#     return Subset(ds, idx)

# def split_train_val(n: int, val_frac: float, seed: int):
#     rng = np.random.default_rng(seed)
#     order = np.arange(n); rng.shuffle(order)
#     n_val = int(round(val_frac * n))
#     val_idx = order[:n_val]; tr_idx = order[n_val:]
#     return tr_idx.tolist(), val_idx.tolist()

# def count_params(model):
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)

# rows = []
# set_seed(SEED)

# for cov in COVERAGES:
#     paths_map = COV_TO_FILES[cov]  # {genome: path}
#     for fold, test_genome in enumerate(GENOME_KEYS, start=1):
#         test_paths = [paths_map[test_genome]]
#         train_paths = [paths_map[g] for g in GENOME_KEYS if g != test_genome]

#         # Build datasets
#         train_full = HyperbubbleDataset(train_paths)
#         test_full  = HyperbubbleDataset(test_paths)

#         # labeled subsets only
#         train_lab = labeled_subset(train_full)
#         test_lab  = labeled_subset(test_full)
#         if len(train_lab) == 0 or len(test_lab) == 0:
#             raise ValueError(f"No labeled samples for cov{cov}, test={test_genome}")

#         # split train->train/val
#         tr_idx, va_idx = split_train_val(len(train_lab), VAL_FRAC, SEED)
#         train_ds = Subset(train_lab, tr_idx)
#         val_ds   = Subset(train_lab, va_idx)
#         test_ds  = test_lab

#         train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
#         val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
#         test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#         print(f"\n[cov={cov} | fold={fold} | test={test_genome}] "
#               f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

#         tag = f"cov{cov}_fold{fold}_{test_genome}_{VARIANT}"
#         ckpt = OUTDIR / f"{tag}.pth"
#         log_csv = OUTDIR / f"{tag}_history.csv"

#         def build_model():
#             return HyperbubbleGNN(
#                 vocab_size=5,
#                 embed_dim=BEST_EMB,
#                 gcn_hidden=BEST_GCN,
#                 edge_hidden=BEST_EDGE,
#                 edge_feat_dim=5,
#                 use_lstm=USE_LSTM,
#             )

#         # Train (early stopping inside fit_variant)
#         best_state, hist_df = fit_variant(
#             build_model_fn=build_model,
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=device,
#             out_ckpt=ckpt,
#             epochs=EPOCHS,
#             lr=LR,
#             weight_decay=WEIGHT_DECAY,
#             patience=PATIENCE,
#             grad_clip=GRAD_CLIP,
#             seed=SEED,
#             print_every=PRINT_EVERY,
#         )
#         hist_df["coverage"] = cov
#         hist_df["fold"] = fold
#         hist_df["test_genome"] = test_genome
#         hist_df["variant"] = VARIANT
#         hist_df.to_csv(log_csv, index=False)

#         # Test
#         model = build_model().to(device)
#         model.load_state_dict(best_state); model.eval()
#         stats = eval_rich_metrics(model, test_loader, device)
#         stats.update({
#             "coverage": cov,
#             "fold": fold,
#             "test_genome": test_genome,
#             "variant": VARIANT,
#             "params": count_params(model),
#         })
#         rows.append(stats)
#         print(f"[TEST] cov={cov} fold={fold} {VARIANT} -> "
#               f"acc_bub={stats['acc_bub']:.4f} acc_dec={stats['acc_dec']:.4f} mrr={stats['mrr']:.4f}")

# # Aggregate & save
# df_cov = pd.DataFrame(rows).sort_values(["coverage","fold","acc_bub"], ascending=[True,True,False])
# df_cov.to_csv(AGG_PATH, index=False)
# print("\nSaved:", AGG_PATH.resolve())
# df_cov.head(12)


In [ ]:
# def plot_bars(df: pd.DataFrame, x_col: str, y_cols, title: str, ylabel: str, fname: Path):
#     ax = df.set_index(x_col)[y_cols].plot(kind="bar", rot=0)
#     ax.set_title(title); ax.set_ylabel(ylabel)
#     ax.figure.tight_layout(); ax.figure.savefig(str(fname), dpi=200); plt.close(ax.figure)

# def plot_lines(x, series: Dict[str, list], title: str, xlabel: str, ylabel: str, fname: Path):
#     plt.figure()
#     for name, ys in series.items():
#         plt.plot(x, ys, marker="o", label=name)
#     plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
#     plt.legend(); plt.tight_layout(); plt.savefig(str(fname), dpi=200); plt.close()

# # 1) Per-coverage: best variant across folds (mean ± std)
# summary = (
#     df_cov
#     .groupby(["coverage","variant"], as_index=False)
#     .agg(acc_bub_mean=("acc_bub","mean"),
#          acc_bub_std=("acc_bub","std"),
#          acc_dec_mean=("acc_dec","mean"),
#          acc_dec_std=("acc_dec","std"),
#          mrr_mean=("mrr","mean"),
#          params=("params","first"))
# )

# # Choose top-5 variants per coverage by acc_bub_mean
# top_variants = (summary.sort_values(["coverage","acc_bub_mean"], ascending=[True,False])
#                       .groupby("coverage").head(5))

# for cov in sorted(df_cov["coverage"].unique()):
#     df_cov_cov = top_variants[top_variants.coverage == cov].copy()
#     if df_cov_cov.empty: 
#         continue
#     fname = OUTDIR / f"cov{cov}_top_variants_bar.png"
#     plot_bars(df_cov_cov, "variant",
#               ["acc_bub_mean","acc_dec_mean","mrr_mean"],
#               f"Coverage {cov}: top variants", "Score", fname)

# # 2) Across coverages: best variant per coverage (by mean acc_bub)
# best_per_cov = (summary.loc[summary.groupby("coverage")["acc_bub_mean"].idxmax()]
#                 .sort_values("coverage"))
# covs = best_per_cov["coverage"].tolist()
# plot_lines(covs,
#            {"Bubble Acc (best)": best_per_cov["acc_bub_mean"].tolist(),
#             "Decision Acc (best)": best_per_cov["acc_dec_mean"].tolist()},
#            "Best-by-coverage accuracy", "Coverage", "Accuracy",
#            OUTDIR / "accuracy_by_coverage_best.png")

# # 3) Params vs accuracy for top-1 per coverage
# plot_lines(best_per_cov["params"].tolist(),
#            {"Bubble Acc": best_per_cov["acc_bub_mean"].tolist(),
#             "Decision Acc": best_per_cov["acc_dec_mean"].tolist()},
#            "Accuracy vs #Params (best per coverage)", "#Params", "Accuracy",
#            OUTDIR / "acc_vs_params_best.png")

# # 4) Optional: save quick pivot for reporting
# pivot = (df_cov.pivot_table(index=["coverage","variant"], values=["acc_bub","acc_dec","mrr"], aggfunc="mean")
#               .reset_index()
#               .sort_values(["coverage","acc_bub"], ascending=[True,False]))
# pivot.to_csv(OUTDIR / "coverage_variant_pivot.csv", index=False)

# print("Plots & pivots saved in:", OUTDIR.resolve())
# pivot.head(10)

In [ ]:
# # Wyjście
# OUTDIR_AUG = Path("aug_cov20_runs"); OUTDIR_AUG.mkdir(parents=True, exist_ok=True)
# AGG_AUG = OUTDIR_AUG / "aug_vs_plain_cov20.csv"

# # Architektura (stała)
# BEST_EMB = 16
# BEST_GCN = 64
# BEST_EDGE = 64
# USE_LSTM = False
# VARIANT_NAME = f"emb{BEST_EMB}_g{BEST_GCN}_e{BEST_EDGE}_{'lstm' if USE_LSTM else 'mean'}"

# # Gdzie szukać danych
# SEARCH_DIRS = ["other_cov_data", "lower_cov_data", "."]

# # Genomy
# GENOME_KEYS = ["ecoli", "klebsiella", "salmonella"]

# def _discover_triplets_cov20_full_upgraded():
#     """
#     Dla każdego genomu zwróć:
#       base:  *_dataset_cov20_full_upgraded*.jsonl          (bez 'augmented' i bez 'val')  -> trening/test
#       aug:   *_dataset_cov20_full_upgraded*_augmented.jsonl -> trening (augmentacja)
#       val:   *_dataset_cov20_full_upgraded*_val.jsonl       -> walidacja
#     """
#     trip = {g: {"base": None, "aug": None, "val": None} for g in GENOME_KEYS}
#     patterns = [
#         "*_dataset_cov20_full_upgraded*_augmented.jsonl",
#         "*_dataset_cov20_full_upgraded*_val.jsonl",
#         "*_dataset_cov20_full_upgraded*.jsonl",
#     ]
#     cands = []
#     for d in SEARCH_DIRS:
#         p = Path(d)
#         for pat in patterns:
#             cands += [str(x) for x in p.glob(pat)]
#     cands = list(dict.fromkeys(cands))

#     def genome_key(name: str):
#         n = Path(name).name.lower()
#         for g in GENOME_KEYS:
#             if g in n:
#                 return g
#         return None

#     for path in cands:
#         name = Path(path).name.lower()
#         g = genome_key(name)
#         if g is None:
#             continue
#         if "augmented" in name:
#             trip[g]["aug"] = path
#         elif "val" in name:
#             trip[g]["val"] = path
#         else:
#             # base: bez 'augmented' i bez 'val'
#             trip[g]["base"] = path

#     missing = {g: [k for k,v in trip[g].items() if not v] for g in GENOME_KEYS}
#     missing = {g: ms for g,ms in missing.items() if ms}
#     if missing:
#         raise FileNotFoundError(f"Brak wymaganych plików: {missing}\nZnalezione: {trip}")
#     return trip

# def set_seed_all(seed:int):
#     import random
#     random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
#     if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# def labeled_subset(ds):
#     idx = []
#     for i in range(len(ds)):
#         d = ds[i]
#         lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
#         if lp >= 0: idx.append(i)
#     return Subset(ds, idx)


In [ ]:
# # %% [AUG-RUNNER v2] 3-fold LOYO: walidacja *_val, test z *_full_upgraded, trening base vs augmented

# set_seed_all(SEED)

# trip = _discover_triplets_cov20_full_upgraded()
# rows = []

# for fold, test_genome in enumerate(GENOME_KEYS, start=1):
#     # --- Ścieżki TEST (tylko genom trzymany na test): base full_upgraded
#     test_paths = [trip[test_genome]["base"]]

#     # --- Ścieżki WALIDACJA (zawsze *_val.jsonl) – z pozostałych, trenowanych genomów
#     val_paths = [trip[g]["val"] for g in GENOME_KEYS if g != test_genome]

#     # --- Dwa warianty treningu: plain vs augmented (z pozostałych genomów)
#     for use_aug in [False, True]:
#         if use_aug:
#             train_paths = [trip[g]["aug"]  for g in GENOME_KEYS if g != test_genome]
#             label_txt = "aug"
#         else:
#             train_paths = [trip[g]["base"] for g in GENOME_KEYS if g != test_genome]
#             label_txt = "plain"

#         # Zbiory danych
#         train_full = HyperbubbleDataset(train_paths)
#         val_full   = HyperbubbleDataset(val_paths)
#         test_full  = HyperbubbleDataset(test_paths)

#         # Tylko etykietowane próbki
#         train_ds = labeled_subset(train_full)
#         val_ds   = labeled_subset(val_full)
#         test_ds  = labeled_subset(test_full)

#         if len(train_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
#             raise ValueError(f"Brak etykiet (fold={fold}, test={test_genome}, {label_txt})")

#         train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
#         val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
#         test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#         print(f"\n[fold {fold} | test={test_genome} | {label_txt}] "
#               f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

#         tag = f"cov20_fold{fold}_{test_genome}_{label_txt}_{VARIANT_NAME}"
#         ckpt = OUTDIR_AUG / f"{tag}.pth"
#         log_csv = OUTDIR_AUG / f"{tag}_hist.csv"

#         def build_model():
#             return HyperbubbleGNN(
#                 vocab_size=5,
#                 embed_dim=BEST_EMB,
#                 gcn_hidden=BEST_GCN,
#                 edge_hidden=BEST_EDGE,
#                 edge_feat_dim=5,
#                 use_lstm=USE_LSTM,
#             )

#         # Trening (early stopping na walidacji *_val)
#         best_state, hist_df = fit_variant(
#             build_model_fn=build_model,
#             train_loader=train_loader,
#             val_loader=val_loader,      # <- używamy osobnego zbioru *_val
#             device=device,
#             out_ckpt=ckpt,
#             epochs=EPOCHS,
#             lr=LR,
#             weight_decay=WEIGHT_DECAY,
#             patience=PATIENCE,
#             grad_clip=GRAD_CLIP,
#             seed=SEED,
#             print_every=5,
#         )

#         # Historia -> CSV
#         hist_df["coverage"] = 20
#         hist_df["fold"] = fold
#         hist_df["test_genome"] = test_genome
#         hist_df["variant"] = VARIANT_NAME
#         hist_df["augmented_train"] = use_aug
#         hist_df.to_csv(log_csv, index=False)

#         # Test na *_full_upgraded genomu trzymanego na test
#         model = build_model().to(device)
#         model.load_state_dict(best_state); model.eval()
#         stats = eval_rich_metrics(model, test_loader, device)
#         stats.update({
#             "coverage": 20,
#             "fold": fold,
#             "test_genome": test_genome,
#             "variant": VARIANT_NAME,
#             "augmented_train": use_aug,
#             "params": sum(p.numel() for p in model.parameters() if p.requires_grad),
#         })
#         rows.append(stats)

#         print(f"  [TEST] {label_txt}: acc_bub={stats['acc_bub']:.4f}  acc_dec={stats['acc_dec']:.4f}  mrr={stats['mrr']:.4f}")

# # Zapis zbiorczego CSV (pod to masz już komórkę agregującą w PL)
# df_aug2 = pd.DataFrame(rows).sort_values(["fold","augmented_train","acc_bub"], ascending=[True, True, False])
# df_aug2.to_csv(AGG_AUG, index=False)

# print("\nZapisano:")
# print(f"  - wyniki zbiorcze: {AGG_AUG.resolve()}")
# print(f"  - checkpointy i historie: {OUTDIR_AUG.resolve()}")


In [ ]:
# # %% [INSIDE-GENOME SWEEP] train/test within the same genome (80/20), best config only (emb16_g64_e64_mean, no LSTM, no augment)

# OUTDIR_IN = Path("inside_genome_runs"); OUTDIR_IN.mkdir(parents=True, exist_ok=True)
# INSIDE_CSV = OUTDIR_IN / "inside_genome_results.csv"

# # Best-fixed architecture (no LSTM, no augmentation)
# BEST_EMB, BEST_GCN, BEST_EDGE = 16, 32, 16
# USE_LSTM = False
# VARIANT_NAME = f"emb{BEST_EMB}_g{BEST_GCN}_e{BEST_EDGE}_{'lstm' if USE_LSTM else 'mean'}"

# # Where to look for per-genome cov20 files; we target "full_upgraded ... ratio_lt_35"
# SEARCH_DIRS = ["other_cov_data", "lower_cov_data", "."]
# GENOME_KEYS = ["ecoli", "klebsiella", "salmonella"]  # extend if you have more

# TRAIN_FRAC = 0.8   # of labeled samples in the genome
# VAL_FRAC_IN_TRAIN = 0.1  # from the train part (for early stopping)

# def set_seed_all(seed:int):
#     import random
#     random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
#     if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# def labeled_subset(ds):
#     idx = []
#     for i in range(len(ds)):
#         d = ds[i]
#         lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
#         if lp >= 0: idx.append(i)
#     return Subset(ds, idx)

# def _discover_cov20_full_upgraded_ratio35() -> Dict[str, str]:
#     """
#     Find exactly one base file per genome:
#       *_dataset_cov20_full_upgraded*ratio_lt_35.jsonl
#     Excludes *_val* and *_augmented* to ensure we use the base dataset.
#     """
#     cands: List[str] = []
#     for d in SEARCH_DIRS:
#         p = Path(d)
#         cands += [str(x) for x in p.glob("*_dataset_cov20_full_upgraded*ratio_lt_35*.jsonl")]
#     # drop duplicates, filter out validation/augmented
#     seen = []
#     filt = []
#     for c in cands:
#         name = Path(c).name.lower()
#         if ("_val" in name) or ("augmented" in name):
#             continue
#         if c not in seen:
#             seen.append(c); filt.append(c)

#     def genome_key(name: str):
#         n = Path(name).name.lower()
#         for g in GENOME_KEYS:
#             if g in n:
#                 return g
#         return None

#     m: Dict[str, str] = {}
#     for path in filt:
#         g = genome_key(path)
#         if g is None: 
#             continue
#         # prefer first match; warn if multiple but keep first
#         if g not in m:
#             m[g] = path
#     missing = [g for g in GENOME_KEYS if g not in m]
#     if missing:
#         raise FileNotFoundError(f"Missing cov20 full_upgraded ratio_lt_35 files for: {missing}\nFound: {m}")
#     return m

# def split_indices(n: int, train_frac: float, seed: int):
#     rng = np.random.default_rng(seed)
#     order = np.arange(n)
#     rng.shuffle(order)
#     n_train = int(round(train_frac * n))
#     train_idx = order[:n_train].tolist()
#     test_idx  = order[n_train:].tolist()
#     return train_idx, test_idx

# def carve_val_from_train(train_idx: List[int], val_frac: float, seed: int):
#     if len(train_idx) == 0 or val_frac <= 0:
#         return train_idx, []
#     rng = np.random.default_rng(seed + 123)
#     order = np.array(train_idx, dtype=int)
#     rng.shuffle(order)
#     n_val = int(round(val_frac * len(order)))
#     val_idx = order[:n_val].tolist()
#     tr_idx  = order[n_val:].tolist()
#     return tr_idx, val_idx

# def count_params(model):
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)

# set_seed_all(SEED)

# genome_to_file = _discover_cov20_full_upgraded_ratio35()
# print("Discovered files per genome:")
# for g, pth in genome_to_file.items():
#     print(f"  - {g}: {pth}")

# rows = []

# for genome in GENOME_KEYS:
#     base_path = genome_to_file[genome]

#     # Single-genome dataset
#     full_ds = HyperbubbleDataset([base_path])
#     lab_ds  = labeled_subset(full_ds)
#     if len(lab_ds) < 5:
#         raise ValueError(f"Too few labeled samples for genome={genome}: {len(lab_ds)}")

#     # 80/20 split on labeled samples
#     tr_idx_all, te_idx = split_indices(len(lab_ds), TRAIN_FRAC, SEED)
#     tr_idx, va_idx = carve_val_from_train(tr_idx_all, VAL_FRAC_IN_TRAIN, SEED)

#     train_ds = Subset(lab_ds, tr_idx)
#     val_ds   = Subset(lab_ds, va_idx)
#     test_ds  = Subset(lab_ds, te_idx)

#     train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
#     val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
#     test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#     print(f"\n[inside-genome] {genome}  train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

#     tag = f"inside_{genome}_{VARIANT_NAME}"
#     ckpt = OUTDIR_IN / f"{tag}.pth"
#     hist_csv = OUTDIR_IN / f"{tag}_history.csv"

#     def build_model():
#         return HyperbubbleGNN(
#             vocab_size=5,
#             embed_dim=BEST_EMB,
#             gcn_hidden=BEST_GCN,
#             edge_hidden=BEST_EDGE,
#             edge_feat_dim=5,
#             use_lstm=USE_LSTM,
#         )

#     best_state, hist_df = fit_variant(
#         build_model_fn=build_model,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         device=device,
#         out_ckpt=ckpt,
#         epochs=EPOCHS,
#         lr=LR,
#         weight_decay=WEIGHT_DECAY,
#         patience=PATIENCE,
#         grad_clip=GRAD_CLIP,
#         seed=SEED,
#         print_every=5,
#     )
#     hist_df["genome"] = genome
#     hist_df["variant"] = VARIANT_NAME
#     hist_df["train_frac"] = TRAIN_FRAC
#     hist_df["val_frac_in_train"] = VAL_FRAC_IN_TRAIN
#     hist_df.to_csv(hist_csv, index=False)

#     model = build_model().to(device)
#     model.load_state_dict(best_state); model.eval()
#     stats = eval_rich_metrics(model, test_loader, device)
#     stats.update({
#         "genome": genome,
#         "variant": VARIANT_NAME,
#         "train_frac": TRAIN_FRAC,
#         "val_frac_in_train": VAL_FRAC_IN_TRAIN,
#         "params": count_params(model),
#         "n_train": len(train_ds),
#         "n_val": len(val_ds),
#         "n_test": len(test_ds),
#     })
#     rows.append(stats)

# # Save combined CSV
# df_inside = pd.DataFrame(rows).sort_values("genome")
# df_inside.to_csv(INSIDE_CSV, index=False)
# print("\nSaved:", INSIDE_CSV.resolve())
# display(df_inside.head())

# # Quick plot: acc_bub/acc_dec per genome
# plt.figure(figsize=(6,4))
# dfp = df_inside.set_index("genome")[["acc_bub","acc_dec"]]
# ax = dfp.plot(kind="bar", rot=0)
# ax.set_ylabel("Accuracy")
# ax.set_title("Inside-genome: acc_bub / acc_dec (train 80% — test 20%)")
# ax.figure.tight_layout()
# plot_path = OUTDIR_IN / "inside_genome_bar.png"
# ax.figure.savefig(plot_path, dpi=200)
# plt.close(ax.figure)
# print("Saved plot:", plot_path.resolve())
